In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
import time

IMAGE_SIZE = 224
BATCH_SIZE = 32

transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_ds = datasets.ImageFolder("dataset/train", transform=transform)
test_ds = datasets.ImageFolder("dataset/test", transform=transform)
val_ds = datasets.ImageFolder("dataset/val", transform=transform)

num_classes = len(train_ds.classes)
print("Classes:", train_ds.classes)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.mobilenet_v2(pretrained=True)
model.classifier[1] = nn.Linear(
    model.classifier[1].in_features,
    num_classes
)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

EPOCHS = 50

def train_epoch(model, loader):
    model.train()
    correct, total_loss = 0, 0

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()

    return total_loss / len(loader.dataset), correct / len(loader.dataset)


def eval_model(model, loader):
    model.eval()
    correct, total_loss = 0, 0

    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * imgs.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()

    return total_loss / len(loader.dataset), correct / len(loader.dataset)

trainm_loss, valm_loss = [], []
trainm_acc, valm_acc = [], []

for epoch in range(EPOCHS):
    tr_loss, tr_acc = train_epoch(model, train_loader)
    val_loss, val_acc = eval_model(model, val_loader)
    trainm_loss.append(tr_loss)
    valm_loss.append(val_loss)
    trainm_acc.append(tr_acc)
    valm_acc.append(val_acc)
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] "
          f"Train Acc: {tr_acc:.4f} : Train Loss: {tr_loss:.4f} | Val Acc: {val_acc:.4f} : Val Loss: {val_loss:.4f}")

In [ ]:
def model_summary(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print("Model Summary")
    print(f"Total parameters     : {total_params:,}")
    print(f"Trainable parameters : {trainable_params:,}")
    param_size = 0
    
    for param in model.parameters():
        param_size += param.nelement() * param.element_size()
    buffer_size = 0
    for buffer in model.buffers():
        buffer_size += buffer.nelement() * buffer.element_size()
    model_size_mb = (param_size + buffer_size) / 1024**2
    print(f"Model size (MB)      : {model_size_mb:.2f} MB")
    
model_summary(model)

In [ ]:
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

print(f"Trainable params: {trainable:,}")
print(f"Total params: {total:,}")

In [ ]:
test_loss, test_acc = eval_model(model, test_loader)
valoss, vaacc = eval_model(model,val_loader)

print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test Loss : {test_loss:.4f}")
print(f"Validation Accuracy : {vaacc:.4f}")
print(f"Validation Loss: {valoss:.4f}")
torch.save(model.state_dict(), "mobilenet_v2.pth")
print("Model saved")

In [ ]:
import matplotlib.pyplot as plt

t_acc = trainm_acc
t_loss = trainm_loss
v_acc = valm_acc
v_loss = valm_loss

plt.figure(figsize=(7, 5))
plt.plot(t_acc, label="Train Accuracy")
plt.plot(v_acc, label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy")
plt.legend()
plt.grid(True)
plt.savefig("Accuracy curve MobileNetv2.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()
print("Accuracy plots saved and displayed")

plt.figure(figsize=(7, 5))
plt.plot(t_loss, label="Train Loss")
plt.plot(v_loss, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True)
plt.savefig("loss curve MobileNetv2.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()
print("Loss plots saved and displayed")


plt.figure(figsize=(7, 5))
plt.plot(t_acc, label="Train Accuracy")
plt.plot(t_loss, label="Train Loss")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training => Accuracy vs Loss")
plt.legend()
plt.grid(True)
plt.savefig("Training Accuracy Vs Loss MobileNetv2.png", dpi=300, bbox_inches="tight")
plt.show() 
plt.close()
print("Accuracy & Loss plots saved and displayed")


plt.figure(figsize=(7, 5))
plt.plot(v_acc, label="Validation Accuracy")
plt.plot(v_loss, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Validation => Accuracy vs Loss")
plt.legend()
plt.grid(True)
plt.savefig("Validation Accuracy Vs Loss MobileNetv2.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()
print("Accuracy & Loss plots saved and displayed")


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

model.eval()

y_true, y_pred = [], []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        labels = labels.to(device)
        outputs = model(imgs)
        preds = outputs.argmax(1)
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())
report = classification_report(
    y_true,
    y_pred,
    target_names=test_ds.classes
)

print("\n===== Classification Report =====\n")
print(report)

with open("classification_report_mv2.txt", "w") as f:
    f.write(report)

print("Classification report saved")


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_true, y_pred)

n_classes=cm.shape[0]
class_names = test_ds.classes


TP = np.zeros(n_classes, dtype=int)
FP = np.zeros(n_classes, dtype=int)
FN = np.zeros(n_classes, dtype=int)
TN = np.zeros(n_classes, dtype=int)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=test_ds.classes,
    yticklabels=test_ds.classes
)

for i in range(n_classes):
    TP[i] = cm[i, i]
    FP[i] = cm[:, i].sum() - TP[i]
    FN[i] = cm[i, :].sum() - TP[i]
    TN[i] = cm.sum() - (TP[i] + FP[i] + FN[i])


plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")

plt.savefig("24-01-26_confusion_matrix_mv2.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()

print("Confusion matrix saved and displayed")

for i in range(n_classes):
    print(f"Class {class_names[i]}:>>>>>>>> TP={TP[i]}, FP={FP[i]}, FN={FN[i]}, TN={TN[i]}")

In [ ]:
import matplotlib.pyplot as plt
import torch
import numpy as np
import random

mean = np.array([0.485, 0.456, 0.406])
std  = np.array([0.229, 0.224, 0.225])

def denormalize(img_tensor):
    img = img_tensor.permute(1, 2, 0).cpu().numpy()
    img = std * img + mean
    img = np.clip(img, 0, 1)
    return img

model.eval()
N = 10
indices = random.sample(range(len(test_ds)), N)

plt.figure(figsize=(20, 8))

for i, idx in enumerate(indices):
    
    img, true_label = test_ds[idx]
    input_tensor = img.unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(input_tensor)
        pred_label = output.argmax(1).item()

    img_np = denormalize(img)
    plt.subplot(2, 5, i + 1)
    plt.imshow(img_np)
    plt.title(
        f"True: {test_ds.classes[true_label]}\n"
        f"Pred: {test_ds.classes[pred_label]}",
        fontsize=11,
        color="black"
    )
    plt.axis("off")

plt.suptitle("Mobilenetv2 – True vs Predicted Labels (Test Set)", fontsize=18)
plt.tight_layout()
plt.savefig("Prediction mv2.png", dpi=300, bbox_inches="tight")
plt.show()